---
title: "Data shape changes under Neural Networks"
author: "Andratx Bellmunt"
abstract: >
  Short notebook to demonstrate to fellow data science colleagues how data shape changes as it is transformed 
  by successive layers of a neural network. A small section on persistent homology is included. Inspired by the paper "Topology of deep neural networks" available at https://arxiv.org/abs/2004.06093.
format:
  html:
    code-fold: true
    self-contained: true
jupyter: python3
number-sections: false
---

# Initialization

## Library imports

In [ ]:
# General imports
from random import random
import numpy as np

In [ ]:
# Neural networks
import tensorflow as tf
from tensorflow import keras as ks

In [ ]:
# Principal component analysis
from sklearn.decomposition import PCA

In [ ]:
# Persistent homology
from ripser import ripser

In [ ]:
# Plotting
import plotly.graph_objects as go
import plotly.io as pio
import teaspoon.TDA.Draw as Draw
import matplotlib.pyplot as plt

In [ ]:
# Set renderer for plots
pio.renderers.default = "plotly_mimetype+notebook_connected"

## Auxiliary functions

In [ ]:
def apply_weights(model, train_data):
    trans_train_data = [train_data]
    for i in range(len(model.layers) - 1):
        linear_part = model.layers[i].get_weights()[0]
        affine_part = model.layers[i].get_weights()[1]
        last = trans_train_data[-1]
        trans_train_data.append(
            ks.layers.ReLU()(np.dot(last, linear_part) + affine_part))

    return np.array(trans_train_data[1:])

In [ ]:
def apply_pca(trans_train_data, dim=2):
    pca = PCA(n_components=dim)
    num_layers = trans_train_data.shape[0]
    pca_trans_train_data = np.array(
        [pca.fit_transform(trans_train_data[i]) for i in range(num_layers)]
    )

    return pca_trans_train_data

In [ ]:
def plot_persistent_homology(dcls, cs, num_layers):
    td = dcls[cs]["td"]
    td_diagrams = dcls[cs]["td_diagrams"]
    pttd = dcls[cs]["pttd"][num_layers-1]
    pttd_diagrams = dcls[cs]["last_pttd_diagrams"]

    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(20,12))

    # Plot data points cloud
    plt.sca(axes[0, 0])
    plt.title("Training data")
    plt.scatter(td[:,0], td[:,1], marker=".")

    # Plot 0-dim diagram
    plt.sca(axes[0,1])
    plt.title("0-dim diagram")
    Draw.drawDgm(td_diagrams[0])

    # Plot 1-dim diagram
    plt.sca(axes[0,2])
    plt.title("1-dim diagram")
    Draw.drawDgm(td_diagrams[1])

    # Plot data points cloud
    plt.sca(axes[1,0])
    plt.title("Transformed data")
    plt.scatter(pttd[:,0], pttd[:,1], marker=".")

    # Plot 0-dim diagram
    plt.sca(axes[1,1])
    plt.title("0-dim diagram")
    Draw.drawDgm(pttd_diagrams[0])

    # Plot 1-dim diagram
    plt.sca(axes[1,2])
    plt.title("1-dim diagram")
    try:
        Draw.drawDgm(pttd_diagrams[1])
    except:
        axes[1,2].plot([0, 1], [0, 1], transform=axes[1,2].transAxes)
        plt.axis([0,1,0,1])

# Training data

We create a dataset with two classes A (in <span style="color:red">**red**</span>) and B (in <span style="color:green">**green**</span>). We choose a very recognizable shape in order to facilitate visualizing the results:

- Class A is contained within a circle of radius 2.

- Class B determines four isolated "islands" within the circle.

In [ ]:
# Creat a dictionary where we will store all the features of the classes
DCLS = {"A":{}, "B":{}}

In [ ]:
# Generate training data
def add_traning_data(dcls):
    # Initialize lists to store the data points and their class
    train_data = []
    train_labels = []

    for _ in range(10000):
        # Generate a random point in a circle
        r = 4 * random()
        a = 2 * np.pi * random()
        x = r * np.cos(a)
        y = r * np.sin(a)
        train_data.append(np.array([x, y]))

        # Assign to class A or B according to some conditions
        cond_left_eye = (x-np.sqrt(2))**2 + (y-np.sqrt(2))**2 < 1
        cond_right_eye = (x+np.sqrt(2))**2 + (y-np.sqrt(2))**2 < 1
        cond_mouth = (x/2)**2 + (y+2)**2 < 1
        cond_nose = x**2 + y**2 < 0.1
        train_labels.append(int(cond_left_eye or cond_right_eye or cond_mouth or cond_nose))

    # Transform to numpy arrays
    train_data = np.array(train_data)
    train_labels = np.array(train_labels)

    # Store the training data
    dcls["A"]["td"] = train_data[np.where(train_labels==0)]
    dcls["B"]["td"] = train_data[np.where(train_labels==1)]

    # Define the color of each class
    dcls["A"]["col"] = "red"
    dcls["B"]["col"] = "green"

    return dcls, train_data, train_labels

In [ ]:
# Plot training data
def plot_training_data(dcls):
    fig = go.Figure()

    for cs in dcls:
        fig.add_trace(
            go.Scatter(
                x=dcls[cs]["td"][:,0],
                y=dcls[cs]["td"][:,1],
                mode='markers',
                name=f"class_{cs}",
                marker_color=dcls[cs]["col"],
                marker_size=2.5)
            )

    fig.update_layout(
        title="Training data",
        width=600,
        height=600,
        xaxis_range=[-4.5,4.5],
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1
    )

    fig.show()

In [ ]:
DCLS, train_data, train_labels = add_traning_data(DCLS)

In [ ]:
plot_training_data(DCLS)

# Neural network

## Create and train

We create a neural network with:

- 15 dense layers (16 neurons each, ReLU activation)

- A final layer with a single neuron and sigmoid activation to produce a binary classifaction

It is important to have a large number of intermediate layers to observe how the data shape changes as it passes through the layers.

In [ ]:
model = ks.Sequential([
    ks.layers.Dense(16, activation="relu"),  # Layer 1 
    ks.layers.Dense(16, activation="relu"),  # Layer 2
    ks.layers.Dense(16, activation="relu"),  # Layer 3
    ks.layers.Dense(16, activation="relu"),  # Layer 4
    ks.layers.Dense(16, activation="relu"),  # Layer 5
    ks.layers.Dense(16, activation="relu"),  # Layer 6
    ks.layers.Dense(16, activation="relu"),  # Layer 7
    ks.layers.Dense(16, activation="relu"),  # Layer 8
    ks.layers.Dense(16, activation="relu"),  # Layer 9
    ks.layers.Dense(16, activation="relu"),  # Layer 10
    ks.layers.Dense(16, activation="relu"),  # Layer 11
    ks.layers.Dense(16, activation="relu"),  # Layer 12
    ks.layers.Dense(16, activation="relu"),  # Layer 13
    ks.layers.Dense(16, activation="relu"),  # Layer 14
    ks.layers.Dense(16, activation="relu"),  # Layer 15
    ks.layers.Dense(1, activation="sigmoid")
])

In [ ]:
model.compile(
    optimizer=ks.optimizers.Adam(),
    loss=ks.losses.BinaryCrossentropy(),
    metrics=[
        ks.metrics.BinaryAccuracy(),
        ks.metrics.FalseNegatives(),
        ks.metrics.FalsePositives()
    ]
)

We fit the neural network to the training data:

In [ ]:
model.fit(train_data, train_labels, epochs=5, batch_size=1)

## Transform data points

We now store how the data points in the training set are transformed by each of the layers of the neural network and provide a visualization. Since the layers have 16 neurons in intermediate steps the data points live in a 16-dimensional space. In order to visualize it we use PCA to project the result into 2-dimensional spaces that capture the separations as well as possible.

In [ ]:
def transform_data_points(dcls):
    for cs in dcls:
        dcls[cs]["ttd"] = apply_weights(model, dcls[cs]["td"])
        dcls[cs]["pttd"] = apply_pca(dcls[cs]["ttd"])
    
    return dcls

In [ ]:
DCLS = transform_data_points(DCLS)

In [ ]:
def plot_data_shape_evolution(dcls, cs, model):
    num_layers = len(model.layers)-1

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=dcls[cs]["td"][:,0],
            y=dcls[cs]["td"][:,1],
            mode='markers',
            marker_color=dcls[cs]["col"],
            marker_size=2.5
        )
    )

    for i in range(num_layers):
        fig.add_trace(
            go.Scatter(
                x=dcls[cs]["pttd"][i,:,0],
                y=dcls[cs]["pttd"][i,:,1],
                mode='markers',
                marker_color=dcls[cs]["col"],
                marker_size=2.5
            )
        )
    
    steps = []
    for i in range(len(fig.data)):
        step = dict(
            method="update",
            args=[{"visible": [False] * len(fig.data)}],  # layout attribute
        )
        step["args"][0]["visible"][i] = True  # Toggle i'th trace to "visible"
        steps.append(step)

    sliders = [dict(
        active=0,
        currentvalue={"prefix": "Layer: "},
        pad={"t": 50},
        steps=steps
    )]

    fig.update_layout(
        width=600,
        height=600,
        title=f"Evolution of class {cs}",
        sliders=sliders
    )

    fig.show()

In [ ]:
plot_data_shape_evolution(DCLS, "A", model)

In [ ]:
plot_data_shape_evolution(DCLS, "B", model)

As we move the slider to the right we see how the data progressively loses its original shape and bends over itself up to the point where, at the end it defines a very clear single 1-dimensional shape. Note also how the x-axis stretches as we take new steps.


# Quick intro to Betti numbers (via persistent homology)

## Betti numbers

Betti numbers are a mathematical concept from the are of Topology. They encode some characteristics of shape:

- The 0th Betti number is the number of connected components of the shape

- The 1st Betti number counts the number of loops (or holes) in the shape

- In general the n-th Betti number counts the number of "n-dimensional loops" in the shape

In [ ]:
fig = go.Figure(
    go.Scatter(
        x=dcls["A"]["td"][:,0],
        y=dcls["A"]["td"][:,1],
        mode='markers',
        name=f"class_{cs}",
        marker_color=dcls["A"]["col"],
        marker_size=2.5
    )
)

fig.update_layout(
    width=600,
    height=600,
    xaxis_range=[-4.5,4.5],
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1
)

fig.show()

In [ ]:
fig = go.Figure(
    go.Scatter(
        x=dcls["B"]["td"][:,0],
        y=dcls["B"]["td"][:,1],
        mode='markers',
        name=f"class_{cs}",
        marker_color=dcls["B"]["col"],
        marker_size=2.5
    )
)

fig.update_layout(
    width=600,
    height=600,
    xaxis_range=[-4.5,4.5],
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1
)

fig.show()


In our example, we are in dimension 2, so the only Betti numbers that are relevant are the 0th and the 1st:
- Class A Betti numbers: (1, 4)
- Class B Betti numbers: (4, 0)

## Persistent homology


Nice video on persistent homology: https://www.youtube.com/watch?v=SbsvM4Gcbl0

In [ ]:
for cs in ["A", "B"]:
    dcls[cs]["td_diagrams"] = ripser(dcls[cs]["td"])["dgms"]
    dcls[cs]["last_pttd_diagrams"] = ripser(dcls[cs]["pttd"][num_layers-1])["dgms"]

In [ ]:
plot_persistent_homology(dcls, "A", num_layers)

In [ ]:
plot_persistent_homology(dcls, "B", num_layers)

# Takeaways

- Data clouds shapes can be studied through Betti numbers

- Betti numbers can be computed with persistent homology

- Each new layer on a neural network simplifies Betti numbers (i.e. shape)

- Once shape is simplified, it is easier to perform some tasks, such as classification